In [4]:
import os
import numpy as np
import tensorflow as tf

# 读取Shakespeare文本文件
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# 打印文本的前100个字符
print(f"文本长度: {len(text)}")
print(f"文本前100个字符:\n{text[:100]}")

# 创建字符级别的字典
vocab = sorted(set(text))
print(f"字典大小: {len(vocab)}")
print(f"字典内容: {vocab}")

# 创建字符到索引的映射
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

# 打印映射示例
print("\n字符到索引的映射示例:")
for char in text[:20]:
    print(f"'{char}' -> {char_to_idx[char]}")

# 将文本转换为数字序列
text_as_int = np.array([char_to_idx[c] for c in text]) #把全部文本都变为id
print(f"\n文本转换为数字序列的前20个元素:\n{text_as_int[:20]}")
print(f"将数字序列转回字符:\n{''.join([idx_to_char[idx] for idx in text_as_int[:20]])}")


文本长度: 1115394
文本前100个字符:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
字典大小: 65
字典内容: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

字符到索引的映射示例:
'F' -> 18
'i' -> 47
'r' -> 56
's' -> 57
't' -> 58
' ' -> 1
'C' -> 15
'i' -> 47
't' -> 58
'i' -> 47
'z' -> 64
'e' -> 43
'n' -> 52
':' -> 10
'
' -> 0
'B' -> 14
'e' -> 43
'f' -> 44
'o' -> 53
'r' -> 56

文本转换为数字序列的前20个元素:
[18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56]
将数字序列转回字符:
First Citizen:
Befor


In [5]:
# 定义序列长度和批次大小
import torch
from torch.utils.data import Dataset, DataLoader

seq_length = 100  # 每个样本的序列长度
batch_size = 64   # 每个批次的样本数量

# 创建自定义数据集类
class ShakespeareDataset(Dataset):
    def __init__(self, text_as_int, seq_length):
        """
        初始化莎士比亚数据集
        
        参数:
            text_as_int: 转换为整数索引的文本
            seq_length: 每个序列的长度
        """
        self.text_as_int = text_as_int
        self.seq_length = seq_length
        self.sub_len = seq_length + 1  # 一个样本的总长度（输入序列+目标字符）
        
    def __len__(self):
        """
        返回数据集中样本的总数
        """
        # 计算可能的序列数量，每个序列长度为seq_length+1
        return len(self.text_as_int)//(self.seq_length+1)  # +1是因为要预测下一个字符
        
    def __getitem__(self, idx):
        """
        获取指定索引的样本
        
        参数:
            idx: 样本索引
            
        返回:
            包含输入序列和目标的样本
        """
        # 返回指定位置的序列片段
        return self.text_as_int[idx*self.sub_len:(idx+1)*self.sub_len]

# 定义collate函数，用于处理批次数据
def collate_fct(batch):
    """
    将批次数据处理成模型所需的输入和目标格式
    
    参数:
        batch: 一批样本数据
        
    返回:
        input_batch: 输入序列批次
        target_batch: 目标序列批次
    """
    # 将批次数据转换为PyTorch张量
    batch = torch.tensor(batch)
    # 输入序列是除了最后一个字符的所有字符
    input_batch = batch[:, :-1]
    # 目标序列是除了第一个字符的所有字符（即每个输入字符对应的下一个字符）
    target_batch = batch[:, 1:]
    return input_batch, target_batch

# 创建数据集实例
shakespeare_dataset = ShakespeareDataset(text_as_int, seq_length)

# 创建数据加载器，用于批量加载数据
dataloader = DataLoader(
    shakespeare_dataset,
    batch_size=batch_size,  # 每批次的样本数
    shuffle=True,           # 随机打乱数据
    drop_last=True,         # 丢弃最后一个不完整的批次
    collate_fn=collate_fct  # 使用自定义的批次处理函数
)

# 打印示例，查看输入和目标
for input_batch, target_batch in dataloader:
    print(f"输入批次形状: {input_batch.shape}")  # 应为[batch_size, seq_length]
    print(f"目标批次形状: {target_batch.shape}")  # 应为[batch_size, seq_length]
    
    # 打印第一个批次的输入和目标，用于验证数据格式是否正确
    print(input_batch)
    print(target_batch)
    break  # 只打印第一个批次

# 打印数据集和数据加载器的基本信息
print(f"\n数据集大小: {len(shakespeare_dataset)}")  # 总样本数
print(f"批次数量: {len(dataloader)}")  # 总批次数


输入批次形状: torch.Size([64, 100])
目标批次形状: torch.Size([64, 100])
tensor([[27, 10,  0,  ..., 53, 59, 56],
        [40, 57, 58,  ..., 53, 59,  1],
        [23, 21, 26,  ..., 52, 41, 43],
        ...,
        [ 8,  0,  0,  ..., 51, 43,  1],
        [53, 61,  1,  ..., 43, 57,  1],
        [42,  1, 58,  ..., 58, 47, 52]])
tensor([[10,  0, 13,  ..., 59, 56,  1],
        [57, 58, 47,  ..., 59,  1, 40],
        [21, 26, 19,  ..., 41, 43, 10],
        ...,
        [ 0,  0, 16,  ..., 43,  1, 57],
        [61,  1, 21,  ..., 57,  1, 46],
        [ 1, 58, 46,  ..., 47, 52, 45]])

数据集大小: 11043
批次数量: 172


C:\Users\ArkhamCraft\AppData\Local\Temp\ipykernel_40000\3466928028.py:55: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  batch = torch.tensor(batch)


In [6]:
import torch.nn as nn
import torch.nn.functional as F

# 定义RNN模型
class SimpleRNN(nn.Module):
    """
    简单的RNN模型，用于字符级语言建模
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        """
        初始化RNN模型
        
        参数:
            vocab_size: 词汇表大小
            embedding_dim: 嵌入层维度
            hidden_dim: 隐藏层维度
            num_layers: RNN层数
        """
        super(SimpleRNN, self).__init__()
        
        # 嵌入层，将整数索引映射为密集向量
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # RNN层
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True  # 输入和输出张量的第一个维度是批次大小
        )
        
        # 输出层，将RNN的输出映射到词汇表大小
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None):
        """
        前向传播
        
        参数:
            x: 输入序列，形状为[batch_size, seq_length]
            hidden: 隐藏状态，默认为None
            
        返回:
            output: 每个时间步的输出，形状为[batch_size, seq_length, vocab_size]
            hidden: 最终隐藏状态
        """
        # 将输入通过嵌入层
        embedded = self.embedding(x)  # [batch_size, seq_length, embedding_dim]
        
        # 通过RNN层
        output, hidden = self.rnn(embedded, hidden)  # output: [batch_size, seq_length, hidden_dim]
        
        # 通过全连接层
        output = self.fc(output)  # [batch_size, seq_length, vocab_size]
        
        return output, hidden

# 设置模型参数
vocab_size = len(vocab)  # 词汇表大小
embedding_dim = 256      # 嵌入层维度
hidden_dim = 512         # 隐藏层维度
num_layers = 1           # RNN层数

# 实例化模型
model = SimpleRNN(vocab_size, embedding_dim, hidden_dim, num_layers)
print(f"模型结构:\n{model}")

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 打印模型参数数量
total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数数量: {total_params}")


模型结构:
SimpleRNN(
  (embedding): Embedding(65, 256)
  (rnn): RNN(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=65, bias=True)
)
模型总参数数量: 444225


In [8]:
# 创建一个小批量数据来测试模型
batch_size = 4
seq_length = 100
test_input = torch.randint(0, vocab_size, (batch_size, seq_length))

# 进行前向计算
with torch.no_grad():
    output, hidden = model(test_input)
    
# 打印输出形状
print(f"输入形状: {test_input.shape}")
print(f"输出形状: {output.shape}")

# 验证输出是否符合预期
assert output.shape == (batch_size, seq_length, vocab_size), "输出形状不符合预期"
assert hidden.shape == (1, batch_size, hidden_dim), "隐藏状态形状不符合预期"


print("模型前向计算验证成功！")

输入形状: torch.Size([4, 100])
输出形状: torch.Size([4, 100, 65])
模型前向计算验证成功！


In [ ]:
# 训练函数
def train_step(model, input_batch, target_batch, optimizer, criterion):
    """
    执行一步训练
    
    参数:
        model: 神经网络模型
        input_batch: 输入批次数据
        target_batch: 目标批次数据
        optimizer: 优化器
        criterion: 损失函数
    
    返回:
        当前批次的损失值
    """
    model.train()  # 设置模型为训练模式
    optimizer.zero_grad()  # 清除之前的梯度
    
    # 前向传播
    output, _ = model(input_batch)  # 获取模型输出，忽略隐藏状态
    
    # 计算损失
    # 重塑输出和目标以适应CrossEntropyLoss
    output = output.reshape(-1, vocab_size)  # 将输出重塑为[batch_size*seq_length, vocab_size]
    target_batch = target_batch.reshape(-1)  # 将目标重塑为[batch_size*seq_length]
    
    loss = criterion(output, target_batch)  # 计算损失
    
    # 反向传播
    loss.backward()  # 计算梯度
    optimizer.step()  # 更新参数
    
    return loss.item()  # 返回损失值（标量）

# 生成文本函数
def generate_text(model, start_string, char2idx, idx2char, num_generate=1000, temperature=1.0):
    """
    使用训练好的模型生成文本
    
    参数:
        model: 训练好的模型
        start_string: 生成文本的起始字符串
        char2idx: 字符到索引的映射字典
        idx2char: 索引到字符的映射字典
        num_generate: 要生成的字符数量
        temperature: 控制采样随机性的参数，较低的值使文本更确定，较高的值增加随机性
    
    返回:
        生成的文本字符串
    """
    model.eval()  # 设置模型为评估模式
    
    # 将起始字符串转换为数字
    input_eval = [char2idx[s] for s in start_string]  # 将每个字符转换为对应的索引
    input_eval = torch.tensor([input_eval], dtype=torch.long)  # 转换为张量，形状为[1, len(start_string)]
    
    # 空字符串用于存储结果
    text_generated = []
    
    # 隐藏状态形状: (1, batch_size, rnn_units)
    hidden = None  # 初始隐藏状态为None，模型会自动初始化
    
    for i in range(num_generate):
        # 前向传播
        predictions, hidden = model(input_eval, hidden)  # 获取预测和更新后的隐藏状态
        
        # 使用最后一个时间步的预测
        predictions = predictions[:, -1, :] / temperature  # 获取最后一个字符的预测并应用温度缩放
        
        # 使用分类分布采样预测的字符
        probabilities = F.softmax(predictions, dim=-1)  # 将logits转换为概率
        predicted_id = torch.multinomial(probabilities, num_samples=1)  # 根据概率分布采样一个字符索引
        
        # 将预测的字符作为下一个输入
        input_eval = predicted_id  # 更新输入为预测的字符索引
        
        # 添加预测的字符到生成的文本
        text_generated.append(idx2char[predicted_id.item()])  # 将索引转换回字符并添加到结果中
    
    return start_string + ''.join(text_generated)  # 返回起始字符串加上生成的文本